# AGENT-P example run

**AGENT-P** runs transcriptome-wide association studies (TWAS) of brain phenotypes. It predicts genetically regulated gene expression (GREx) with four model sets — PrediXcan (`PDX`), UTMOST (`UTM`), JTI (`JTI`), FUSION (`FUS`) — in eight brain regions, and tests GREx for association with a phenotype in two ways:

* **Summary TWAS**: from GWAS summary statistics
* **Individual TWAS**: from individual-level genotypes, covariates, and phenotypes

This notebook runs both on the example data in the AGENT-P repository, using the **JTI caudate** models, and plots the results. It takes about 15–20 minutes on a free Colab runtime: **Runtime → Run all**.

| Step | What it does |
|---|---|
| 1. Install | AGENT-P and the tools it calls (plink2, bgenix) |
| 2. Project | load the example cohort and QC its genotypes |
| 3. Summary TWAS | GWAS of amygdala volume + JTI caudate models |
| 4. Individual TWAS | predict caudate GREx for each subject, then test amygdala volume |
| 5. Results | top genes from both TWAS |
| 6. Plots | Manhattan and Miami plots |

## 1. Install

AGENT-P runs in its own conda environment (Python 3.10 with plink2 and bgenix), separate from Colab's own Python. This cell downloads the AGENT-P code from GitHub and creates the environment with [micromamba](https://mamba.readthedocs.io) (about 3–5 minutes).

In [ ]:
%%bash
set -e
BRANCH=agentp-update-20261105

## AGENT-P code and example data (the full models in Git LFS are not needed here)
[ -d agentp ] || GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 -b $BRANCH https://github.com/nhunghoang/agentp.git

## conda environment with micromamba
export MAMBA_ROOT_PREFIX=$PWD/mamba
[ -x bin/micromamba ] || curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
[ -d agentp_env ] || bin/micromamba create -y -q -p $PWD/agentp_env -c conda-forge -c bioconda --override-channels \
    python=3.10 "numpy<2" pandas "scipy>=1.11" statsmodels h5py patsy bitarray matplotlib bgen-reader pyliftover pip \
    plink2=2.00a5.12 bgenix=1.1.7

## AGENT-P itself (--no-deps keeps the conda packages)
agentp_env/bin/pip install -q --no-deps -e agentp
agentp_env/bin/python -c "import agentp; print('AGENT-P installed from', agentp.__path__[0])"

The next cell defines `%%agentp`: a cell that starts with `%%agentp` runs with AGENT-P's Python, from the `agentp/example` folder (which holds `example_data/` and `example_models/`). Each such cell is a separate Python process, so each starts by importing AGENT-P and selecting the example models.

In [ ]:
import os, subprocess
from IPython.core.magic import register_cell_magic

ROOT = os.getcwd()
ENV = f'{ROOT}/agentp_env'
EXAMPLE = f'{ROOT}/agentp/example'

## AGENT-P's environment first on PATH (python, plink2, bgenix); no Colab Python settings
RUN_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'MPLBACKEND')}
RUN_ENV.update(PATH=f'{ENV}/bin:' + os.environ['PATH'], MPLBACKEND='Agg')

@register_cell_magic
def agentp(line, cell):
    '''Run this cell with AGENT-P's Python, from the example folder, and show its output.'''
    proc = subprocess.Popen([f'{ENV}/bin/python', '-u', '-c', cell], cwd=EXAMPLE, env=RUN_ENV,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for out in proc.stdout:
        print(out, end='')
    if proc.wait() != 0:
        raise RuntimeError('This AGENT-P step failed: see the output above')

print(subprocess.run('plink2 --version; bgenix -help | head -1', shell=True, env=RUN_ENV,
                     capture_output=True, text=True).stdout)

## 2. Create the project

A **project** is a folder that holds the inputs and results of one analysis. The example cohort has 2,000 subjects:

* `subjects.txt`: subject IDs
* `covariates.csv`: age, sex, and 3 genetic principal components
* `volumes.csv`: four brain volumes (putamen, amygdala, hippocampus, caudate)
* `genotypes/test_c*.bgen`: genotypes for chromosomes 1–22, which AGENT-P quality-controls with plink2

`set_models_dir('example_models')` selects the small example models (caudate only) that come with the repository; real analyses use the full models (see the README).

In [ ]:
%%agentp
from agentp import Project, set_models_dir
set_models_dir('example_models')

project = Project('example_run')
project.set_subjects('example_data/subjects.txt')
project.add_covariates('example_data/covariates.csv')
project.add_phenotypes('example_data/volumes.csv')
project.add_genotypes('example_data/genotypes', 'test_c*.bgen')

## 3. Summary TWAS

Summary TWAS combines GWAS summary statistics with the GREx model weights and the LD between the model SNPs. Here, the input is a GWAS of amygdala volume in the example cohort (`amygdala_volume_gwas.regenie.gz`, made with REGENIE), and the models are the JTI caudate models.

In [ ]:
%%agentp
from agentp import Project, SummaryTWAS, set_models_dir
set_models_dir('example_models')
project = Project('example_run')

stwas = SummaryTWAS(project, 'JTI', 'caudate')
stwas.add_gwas('amygdala_volume', 'example_data/amygdala_volume_gwas.regenie.gz')
stwas.run_twas('amygdala_volume')

## 4. Individual TWAS

Individual TWAS first predicts caudate GREx for every subject from their genotypes (`predict_grex`), then tests each gene's GREx for association with amygdala volume, adjusting for the covariates (`run_twas`). This is the longest step (about 10 minutes on Colab).

In [ ]:
%%agentp
from agentp import Project, IndividualTWAS, set_models_dir
set_models_dir('example_models')
project = Project('example_run')

itwas = IndividualTWAS(project, 'JTI', 'caudate')
itwas.predict_grex()
itwas.run_twas('amygdala_volume')

## 5. Results

Each TWAS writes one table per phenotype: `example_run/sTWAS/JTI_caudate/amygdala_volume.csv` and `example_run/iTWAS/JTI_caudate/amygdala_volume.csv`. Below, the top genes of the summary TWAS next to their individual TWAS results.

In [ ]:
import pandas as pd

res = {t: pd.read_csv(f'{EXAMPLE}/example_run/{t}/JTI_caudate/amygdala_volume.csv', sep='\t')
       for t in ['sTWAS', 'iTWAS']}
for t, name in [('sTWAS', 'Summary TWAS'), ('iTWAS', 'Individual TWAS')]:
    df = res[t]
    bonf = 0.05 / len(df)
    print(f'{name}: {len(df)} genes tested, {(df.pvalue < bonf).sum()} with p < {bonf:.1e} (Bonferroni)')

top = res['sTWAS'][['gene', 'gene_name', 'zscore', 'pvalue']] \
        .merge(res['iTWAS'][['gene', 'zscore', 'pvalue']], on='gene', suffixes=(' (summary)', ' (individual)')) \
        .sort_values('pvalue (summary)').head(10).reset_index(drop=True)
top

With only 2,000 subjects, no gene passes the Bonferroni threshold in this example, but the two TWAS agree on the strongest genes (the summary TWAS uses a GWAS of the same cohort).

## 6. Plots

* **Manhattan plot**: individual TWAS p-values by gene position; the dashed line is the Bonferroni threshold.
* **Miami plot**: summary TWAS (top) mirrored against individual TWAS (bottom).

In [ ]:
%%agentp
import os
from agentp import load_pvalues, manhattan_plot, miami_plot

s = load_pvalues('example_run/sTWAS/JTI_caudate/amygdala_volume.csv')
i = load_pvalues('example_run/iTWAS/JTI_caudate/amygdala_volume.csv')

os.makedirs('example_run/figures', exist_ok=True)
manhattan_plot(i, label='Individual TWAS (JTI caudate)', title='amygdala_volume',
               out_file='example_run/figures/manhattan.png')
miami_plot(s, i, top_label='Summary TWAS (JTI caudate)', bottom_label='Individual TWAS (JTI caudate)',
           title='amygdala_volume', out_file='example_run/figures/miami.png')

In [ ]:
from IPython.display import Image, display
display(Image(f'{EXAMPLE}/example_run/figures/manhattan.png', width=900))
display(Image(f'{EXAMPLE}/example_run/figures/miami.png', width=900))

## Next steps

* **Other models**: replace `'JTI'` with `'PDX'` or `'UTM'` (FUSION, `'FUS'`, also needs R). The example models cover the caudate only; the full models cover eight brain regions.
* **Your own data**: download the full models and follow the README (`environment.yml` sets up the same environment locally).
* **More than one model set**: `run_all_twas` runs every model set and region, and `collect_twas` combines them per gene with ACAT.